In [26]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.layers import Dense
from keras import Sequential
from lazypredict.Supervised import LazyRegressor
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [27]:
df = pd.read_csv('bayut_listings_cleaned.csv')
df.head()

,Price,Property Type,Number Of Bedrooms,Number Of Bathrooms,Area,Location
0,1650000,FLOOR,5,4,523,OTHER
1,1249000,FLOOR,6,5,500,OTHER
2,499000,APARTMENT,3,3,154,OTHER
3,499000,APARTMENT,3,3,154,OTHER
4,499000,APARTMENT,3,3,154,OTHER


In [28]:
location_Enc = df['Location']
y = df['Price']


In [29]:
df.head(1)

,Price,Property Type,Number Of Bedrooms,Number Of Bathrooms,Area,Location
0,1650000,FLOOR,5,4,523,OTHER


In [ ]:
targetenc = TargetEncoder(smooth='auto', shuffle=False)
df['Location'] = targetenc.fit_transform(df[['Location']],y)


In [31]:
X_encoded = pd.get_dummies(df,columns=['Property Type'],dtype=int)
X_encoded.head()

,Price,Number Of Bedrooms,Number Of Bathrooms,Area,Location,Property Type_APARTMENT,Property Type_FLOOR,Property Type_REST HOUSE,Property Type_VILLA
0,1650000,5,4,523,0.00,0,1,0,0
1,1249000,6,5,500,0.00,0,1,0,0
2,499000,3,3,154,0.00,1,0,0,0
3,499000,3,3,154,0.00,1,0,0,0
4,499000,3,3,154,0.00,1,0,0,0


In [32]:
x_train,x_test,y_train,y_test = train_test_split(X_encoded,y,test_size=0.2,random_state=32)

In [33]:
# model_linear = LinearRegression().fit(x_train,y_train)
# model_bay = BayesianRidge().fit(x_train,y_train)
# model_lasso = Lasso().fit(x_train,y_train)
# print(f'model performance on training data: {model_linear.score(x_train,y_train)} \nmodel performance on testing data: {model_linear.score(x_test,y_test)}')
# print(f'model_bay performance on training data: {model_bay.score(x_train,y_train)} \nmodel_bay performance on testing data: {model_bay.score(x_test,y_test)}')
# print(f'model_lasso performance on training data: {model_lasso.score(x_train,y_train)} \nmodel_lasso performance on testing data: {model_lasso.score(x_test,y_test)}')

In [34]:
# model = tf.keras.Sequential([Dense(units=25,activation='relu'),
#                              Dense(units=15,activation='relu'),
#                              Dense(units=1,activation= tf.keras.activations.extratreesregressor)])
# model.compile(loss=tf.keras.losses.MeanSquaredError(
#     reduction='sum_over_batch_size',
#     name='mean_squared_error'),
#     metrics=[tf.keras.metrics.MeanSquaredError()]
# )
# history = model.fit(
#     x_train,
#     y_train,
#     batch_size=64,
#     epochs=100,
#     # We pass some validation for
#     # monitoring validation loss and metrics
#     # at the end of each epoch
   
# )

In [35]:
# print(history.history) # this will print a dictionary object, now you need to grab the metrics / score you're looking for


In [36]:
# loss, accuracy = model.evaluate(x_train, x_train, batch_size=1000)
# print(f"Test Loss: {loss:.4f}")
# print(f"Test Accuracy: {accuracy:.4f}")

In [37]:
# reg = LazyRegressor(verbose=0,ignore_warnings=False, custom_metric=None )
# models,predictions = reg.fit(x_train, x_test, y_train, y_test)
# models.head()

In [38]:
neigh = KNeighborsRegressor(n_neighbors=1)
neigh.fit(x_train,y_train)
neigh.score(x_train,y_train)

1.0

In [39]:
neigh.score(x_test,y_test)

0.9991597323510718

In [40]:
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 5, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

pipeline = Pipeline([
    # ('scalar', StandardScaler()),
     ('poly', PolynomialFeatures(degree=2)),
    ('model', RandomForestRegressor(random_state=42))
])

grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

print("Best parameters:", grid.best_params_)
print("Train score:", grid.score(x_train, y_train))
print("Test score:", grid.score(x_test, y_test))


Fitting 5 folds for each of 90 candidates, totalling 450 fits
Best parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Train score: 0.9927936438323349
Test score: 0.9877633550072696


In [41]:
param_grid = {
    'model__n_estimators': [200],
    'model__max_depth': [23],
    'model__min_samples_split': [4],
    'model__min_samples_leaf': [9],
}

pipeline = Pipeline([
    # ('scalar', StandardScaler()),
     ('poly', PolynomialFeatures(degree=2)),
    ('model', RandomForestRegressor())
])

grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1,scoring='r2')
grid.fit(x_train, y_train)

print("Best parameters:", grid.best_params_)
print("Train score:", grid.score(x_train, y_train))
print("Test score:", grid.score(x_test, y_test))

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best parameters: {'model__max_depth': 23, 'model__min_samples_leaf': 9, 'model__min_samples_split': 4, 'model__n_estimators': 200}
Train score: 0.9049493757906262
Test score: 0.9891839985194045


In [42]:
forest = RandomForestRegressor(random_state=42, n_estimators=200,max_depth=23,min_samples_leaf=4,min_samples_split=4).fit(x_train, y_train)
print("Train score:", forest.score(x_train, y_train))
print("Test score:", forest.score(x_test, y_test))

Train score: 0.945647957954729
Test score: 0.9996388535841744


In [43]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMRegressor

# Define parameter grid for LGBMRegressor
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [-1, 5, 10, 20, 30],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__num_leaves': [31, 50, 100],
    'model__min_child_samples': [5, 10, 20]
}

# Pipeline using LightGBM
pipeline = Pipeline([
    # ('scalar', StandardScaler()),  # Optional: may not be needed for tree-based models
    ('poly', PolynomialFeatures(degree=2)),
    ('model', LGBMRegressor(random_state=42))
])

# GridSearchCV with cross-validation
grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

# Output the results
print("Best parameters:", grid.best_params_)
print("Train score:", grid.score(x_train, y_train))
print("Test score:", grid.score(x_test, y_test))


Fitting 5 folds for each of 270 candidates, totalling 1350 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000605 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4443
[LightGBM] [Info] Number of data points in the train set: 11113, number of used features: 48
[LightGBM] [Info] Start training from score 1407520.851345
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': -1, 'model__min_child_samples': 5, 'model__n_estimators': 100, 'model__num_leaves': 31}
Train score: 0.9919859782133907
Test score: 0.9942281975561226


In [44]:

# # Define parameter grid for LGBMRegressor
# param_grid = {
#     'model__n_estimators': [x for x in range(180,220)],
#     'model__max_depth': [x for x in range(15,25)],
#     'model__learning_rate': [0.005,0.0075,0.01],
#     'model__num_leaves': [x for x in range(80,120)],
#     'model__min_child_samples': [x for x in range(15,25)]
# }

# # Pipeline using LightGBM
# pipeline = Pipeline([
#     # ('scalar', StandardScaler()),  # Optional: may not be needed for tree-based models
#     ('poly', PolynomialFeatures(degree=2)),
#     ('model', LGBMRegressor(random_state=42))
# ])

# # GridSearchCV with cross-validation
# grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)
# grid.fit(x_train, y_train)

# # Output the results
# print("Best parameters:", grid.best_params_)
# print("Train score:", grid.score(x_train, y_train))
# print("Test score:", grid.score(x_test, y_test))


In [45]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_neighbors': list(range(1, 30))
}

pipeline = Pipeline([
    ('scalar', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2)),
    ('model', KNeighborsRegressor())
])

grid = GridSearchCV(pipeline, param_grid, cv=5)
grid.fit(x_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation score:", grid.best_score_)
print("Test score:", grid.score(x_test, y_test))


Best parameters: {'model__n_neighbors': 1}
Best cross-validation score: 0.9573320545819097
Test score: 0.970032215225256


In [46]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipeline, x_train, y_train, cv=5)
print("Cross-validation scores:", scores)
print("Average CV score:", scores.mean())

Cross-validation scores: [0.97921385 0.96813107 0.97821352 0.81225721 0.9600447 ]
Average CV score: 0.939572070563534
